# __Exploratory Analysis and Feature Engineering of the NEMSIS 2023 Cardiac Arrest Dataset__

This notebook documents the exploratory analysis, preprocessing, and feature engineering steps performed on the 2023 NEMSIS Cardiac Arrest Subset dataset. The full NEMSIS dataset contains a large number of EMS variables spanning multiple domains of patient care. The full dataset is distributed across multiple relational tables linked by patient care report identifiers (PCR keys). Of the 20 available tables, six were selected and integrated to construct the final analytical dataset used in this project.

The raw dataset consisted of a large number of coded variables, placeholder values, and inconsistently structured categorical fields that required cleaning and consolidation prior to analysis. During exploration, the analysis was narrowed to cardiac arrest-related variables.

This notebook explores the structure and quality of the dataset, identifies clinically relevant variables associated with Return of Spontaneous Circulation (ROSC), and demonstrates the preprocessing workflow used to prepare the final analytical dataset.

Certain variables used coded placeholder values (e.g., 7701001, 7701003) rather than standard null values, requiring domain-specific handling during preprocessing.

The exploratory findings and engineered features developed here informed the focused statistical analyses and predictive modeling performed in [02_analysis.ipynb](./02_analysis.ipynb).

## Reference Documentation and Data Dictionaries

The NEMSIS dataset uses standardized coded variables and reference values defined by official NEMSIS documentation. Interpretation of categorical fields, placeholder values, procedures, medications, and rhythm classifications required consultation of the associated NEMSIS data dictionaries and reference materials.

These references were used extensively during preprocessing and feature engineering to map coded values into clinically interpretable categories.

Relevant reference materials included:
- [NEMSIS Data Dictionary](https://nemsis.org/media/nemsis_v3/release-3.5.0/DataDictionary/PDFHTML/EMSDEMSTATE/index.html)
- [RxNav (Natural Library of Medicine)](https://mor.nlm.nih.gov/RxNav/)
- [NEMSIS V3 Custom Element Library](https://nemsis.org/media/customelementlibrary/index.html#!/details?state=montana&id=eProcedures.03)

Reusable categorical mappings and grouping rules were maintained within the [mappings.py](../src/mappings.py) module to support consistent preprocessing and feature construction.

## Environment Setup and Imports

In [41]:
import pandas as pd
from collections import Counter
import pyreadstat
from functools import reduce
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))
from mappings import *
from processing import clean_missing, get_unique, get_count, get_epi_count, total_count, ROSC_status, simplify_rhythm

## Dataset Acquisition and Initial Loading

The NEMSIS 2023 Cardiac Arrest Subset was obtained from the public [NEMSIS research dataset repository](https://nemsis.org/datasets/). The dataset was loaded and inspected to evaluate overall structure, column composition, and variable encoding prior to preprocessing and feature engineering. Raw NEMSIS tables were provided in SAS7BDAT format and imported using the `pyreadstat` library for preprocessing and analysis within Python.

Key tables included:
- Main cardiac arrest incident data
- EMS medication administration records
- EMS procedure records
- Cardiac arrest resuscitation variables
- ROSC outcome variables
- Initial symptoms

In [2]:
df_main, meta_main = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_1to1.sas7bdat"
)
df_main.shape

(274531, 19)

In [3]:
df_meds, meta_meds = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_emedications.sas7bdat"
)
df_meds.shape

(1150449, 6)

In [4]:
df_proc, meta_proc = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_eprocedures.sas7bdat"
)
df_proc.shape

(1480454, 5)

In [5]:
df_arrest_resus, meta_arrest_03 = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_earrest03.sas7bdat"
)
df_arrest_resus.shape

(519129, 2)

In [6]:
df_arrest_ROSC, meta_arrest_12 = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_earrest12.sas7bdat"
)
df_arrest_ROSC.shape

(283592, 2)

In [7]:
df_symptoms, meta_situation_09 = pyreadstat.read_sas7bdat(
    "../data/CardiacArrestPUD2023/cardiacarrestpud_esituation09.sas7bdat"
)
df_symptoms.shape

(274531, 2)

The following tables were selected from the larger NEMSIS cardiac arrest dataset and used to construct the analytical dataset for this project.

In [8]:
table_data = [
    ["Main Incident Table", df_main.shape[0], df_main.shape[1], "Cardiac arrest event data"],
    ["Medications", df_meds.shape[0], df_meds.shape[1], "Medication administrations"],
    ["Procedures", df_proc.shape[0], df_proc.shape[1], "EMS procedures"],
    ["Resuscitation", df_arrest_resus.shape[0], df_arrest_resus.shape[1], "Resuscitation attempts"],
    ["ROSC", df_arrest_ROSC.shape[0], df_arrest_ROSC.shape[1], "ROSC status"],
    ["Initial Symptoms", df_symptoms.shape[0], df_symptoms.shape[1], "Initial symptoms"]
]
table = pd.DataFrame(table_data, columns=["Table", "Rows", "Columns", "Description"])
display(table.style.hide(axis="index"))

Table,Rows,Columns,Description
Main Incident Table,274531,19,Cardiac arrest event data
Medications,1150449,6,Medication administrations
Procedures,1480454,5,EMS procedures
Resuscitation,519129,2,Resuscitation attempts
ROSC,283592,2,ROSC status
Initial Symptoms,274531,2,Initial symptoms


### Initial Dataset Inspection

Initial inspection of the raw tables demonstrated that the NEMSIS dataset used encoded categorical values and repeated patient care report identifiers (PCR keys). They also showed event-level relational structures across medications and procedures. These observations guided the subsequent preprocessing, decoding, aggregation, and feature engineering steps used to create the analytical dataset.

In [9]:
df_main.head(3)

,PcrKey,eArrest_01,eArrest_02,eArrest_07,eArrest_11,eArrest_16,eArrest_18,eDispatch_01,eDispatch_02,eDisposition_16,eDisposition_19,eDisposition_21,ePatient_15,ePatient_16,eResponse_07,eScene_01,eScene_09,eSituation_02,eSituation_13
0,225614118.0,3001003,3002001,3007001,3011001,3016005,3018003,2301013,7701003,7701001,7701001,7701001,88.0,2516009,2207009,9923003,Y92.12,9922001,7701003
1,225614544.0,3001003,3002001,3007001,3011011,3016009,3018003,2301019,7701003,7701001,7701001,7701001,77.0,2516009,2207009,9923003,Y92.0,9922001,7701003
2,225614812.0,3001003,3002001,7701001,7701001,7701001,7701001,2301013,2302001,4216005,4219003,4221003,65.0,2516009,2207003,9923003,Y92.01,9922001,2813003


In [10]:
df_meds.head(3)

,PcrKey,eMedications_03,eMedications_03Descr,eMedications_05,eMedications_06,eMedications_07
0,225614118.0,7701003,Not Recorded,0.0,7701003,7701003
1,225614544.0,1008377,Calcium Chloride / Lactate / Potassium Chlorid...,500.0,3706025,9916003
2,225614544.0,317361,EPINEPHrine 0.1 MG/ML,1.0,3706021,9916003


In [11]:
df_proc.head(3)

,PcrKey,eProcedures_02,eProcedures_03,eProcedures_06,eProcedures_08
0,225614118.0,9923001,422618004,9923003,9916003
1,225614544.0,9923001,426220008,9923001,9916003
2,225614544.0,9923001,429283006,9923001,9916003


In [12]:
df_arrest_resus.head(3)

,PcrKey,eArrest_03
0,225614118.0,3003005
1,225614544.0,3003005
2,225614812.0,3003003


In [13]:
df_arrest_ROSC.head(3)

,PcrKey,eArrest_12
0,225614118.0,3012001
1,225614544.0,3012001
2,225614812.0,7701001


In [14]:
df_symptoms.head(3)

,PcrKey,eSituation_09
0,225614118.0,I46.9
1,225614544.0,I46.9
2,225614812.0,R09.2


### Selecting Clinically Relevant Variables

The complete dataset contained a substantially larger number of variables than required for this analysis. Variables related to cardiac arrest presentation, EMS interventions, and patient outcomes were selected for further exploration and preprocessing. Columns were renamed for clarity.

In [15]:
df = df_main[["PcrKey", "eResponse_07", "eArrest_01", "eArrest_02", "eArrest_07", "eArrest_11",
              "eArrest_18", "eSituation_13", "eDisposition_19"]]

df_medications = df_meds[["PcrKey", "eMedications_03"]]

df_procedures = df_proc[["PcrKey", "eProcedures_03"]]

In [16]:
df.rename(columns={"eResponse_07": "Response_Type", "eArrest_01": "Arrest", "eArrest_02": "Arrest_Etiology",
                   "eArrest_07": "AED_Prior_to_EMS", "eArrest_11": "Initial_Rhythm", "eArrest_18": "End_of_EMS_Cardiac_Event",
                   "eSituation_13": "Initial_Acuity", "eDisposition_19": "Final_Acuity"}, inplace=True)

df_medications.rename(columns={"eMedications_03": "Medications"}, inplace=True)

df_procedures.rename(columns={"eProcedures_03": "Procedures"}, inplace=True)

df_arrest_resus.rename(columns={"eArrest_03": "Resuscitation"}, inplace=True)

df_arrest_ROSC.rename(columns={"eArrest_12": "ROSC"}, inplace=True)

df_symptoms.rename(columns={"eSituation_09": "Symptoms"}, inplace=True)

### Handling Placeholder Missing Values

Many NEMSIS variables used coded placeholder values rather than standard null values to represent unknown, unrecorded, or unavailable responses. These values were converted into interpretable missing values during preprocessing to improve downstream analysis and feature engineering. For event-level tables such as medications and procedures, rows containing valid PCR identifiers but missing intervention-specific information were removed prior to aggregation and merging. This prevented non-informative even records from inflating intervention counts. Patient-level variables with partial missingness were retained in the final analytical dataset for downstream analysis.

In [17]:
df_clean = df.map(clean_missing)
missing_df = df_clean.isna().sum().sort_values(ascending=False)
missing_df.head()

Final_Acuity                141264
Initial_Acuity               64230
Initial_Rhythm               30873
End_of_EMS_Cardiac_Event     16977
AED_Prior_to_EMS             10161
dtype: int64

In [18]:
df_medications = df_medications.map(clean_missing).dropna()
missing_meds = [
    ["Before dropping null values", df_meds.shape[0]],
    ["After dropping null values", df_medications.shape[0]]
]
mmeds = pd.DataFrame(missing_meds, columns=["", "Rows"])
display(mmeds.style.hide(axis="index"))

,Rows
Before dropping null values,1150449
After dropping null values,1066327


In [19]:
df_procedures = df_procedures.map(clean_missing).dropna()
missing_proc = [
    ["Before dropping null values", df_proc.shape[0]],
    ["After dropping null values", df_procedures.shape[0]]
]
mproc = pd.DataFrame(missing_proc, columns=["", "Rows"])
display(mproc.style.hide(axis="index"))

,Rows
Before dropping null values,1480454
After dropping null values,1435459


In [20]:
df_resuscitation = df_arrest_resus.map(clean_missing).dropna()
missing_resus = [
    ["Before dropping null values", df_arrest_resus.shape[0]],
    ["After dropping null values", df_resuscitation.shape[0]]
]
mresus = pd.DataFrame(missing_resus, columns=["", "Rows"])
display(mresus.style.hide(axis="index"))

,Rows
Before dropping null values,519129
After dropping null values,505886


In [21]:
df_ROSC = df_arrest_ROSC.map(clean_missing).dropna()
missing_ROSC = [
    ["Before dropping null values", df_arrest_ROSC.shape[0]],
    ["After dropping null values", df_ROSC.shape[0]]
]
mROSC = pd.DataFrame(missing_ROSC, columns=["", "Rows"])
display(mROSC.style.hide(axis="index"))

,Rows
Before dropping null values,283592
After dropping null values,261301


In [22]:
df_symptoms = df_symptoms.map(clean_missing)
missing_symptoms = df_symptoms.isna().sum().sort_values(ascending=False)
missing_symptoms.head()

Symptoms    21627
PcrKey          0
dtype: int64

### Variable Decoding and Value Mapping

Many variables within the NEMSIS dataset were stored as encoded categorical values rather than human-readable labels. Relevant variables were mapped to interpretable clinical categories using the associated NEMSIS data dictionaries and reference tables. This preprocessing step improved interpretability, simplified downstream analysis, and enabled clearer visualization of cardiac arrest characteristics and EMS interventions.

Examples of decoded variables included initial cardiac rhythm, AED use prior to EMS arrival, ROSC status, EMS interventions, patient acuity, and medication administration records.

Highly granular symptom and clinical presentation variables were additionally consolidated into broader clinically meaningful categories to reduce sparsity and improve interpretability during downstream analysis.

In [23]:
df_clean["Response_Type"] = df_clean["Response_Type"].map(RESPONSE_TYPE)
df_clean["Arrest"] = df_clean["Arrest"].map(ARREST)
df_clean["Arrest_Etiology"] = df_clean["Arrest_Etiology"].map(ARREST_ETIOLOGY)
df_clean["AED_Prior_to_EMS"] = df_clean["AED_Prior_to_EMS"].map(ARREST_AED_USE_PRIOR_TO_EMS_ARRIVAL)
df_clean["Initial_Rhythm"] = df_clean["Initial_Rhythm"].map(ARREST_FIRST_MONITORED_RHYTHM)
df_clean["End_of_EMS_Cardiac_Event"] = df_clean["End_of_EMS_Cardiac_Event"].map(ARREST_END_OF_EMS_CARDIAC_EVENT)
df_clean["Initial_Acuity"] = df_clean["Initial_Acuity"].map(INITIAL_PATIENT_ACUITY)
df_clean["Final_Acuity"] = df_clean["Final_Acuity"].map(FINAL_PATIENT_ACUITY)

df_medications["Medications"] = df_medications["Medications"].map(MEDICATIONS)

df_procedures["Procedures"] = df_procedures["Procedures"].map(PROCEDURES)

df_resuscitation["Resuscitation"] = df_resuscitation["Resuscitation"].map(ARREST_RESUSCITATION)

df_ROSC["ROSC"] = df_ROSC["ROSC"].map(ARREST_ROSC)

df_symptoms["ICD_Group"] = df_symptoms["Symptoms"].str[0]
df_symptoms["Symptoms"] = df_symptoms["ICD_Group"].map(ICD_CHAPTER_MAP).fillna("Other")
df_symptoms = df_symptoms.drop(columns=["ICD_Group"])

### Aggregation of Event-Level Data by PCR Key

Several NEMSIS tables contained event-level records in which multiple medications, procedures, or symptoms could be associated with a single patient care report (PCR). To construct a patient-level analytical dataset, these records were aggregated by PCR key prior to merging. This process consolidated repeated intervention records into summary features representing medication administration, procedures performed, and intervention counts for each cardiac arrest encounter.

Repeated medications and procedures administered during the same encounter were preserved during aggregation to retain information regarding intervention intensity and resuscitation complexity.

In [24]:
df_medications_lists = df_medications.groupby("PcrKey")["Medications"].apply(list).reset_index(name="Medications")
df_procedures_lists = df_procedures.groupby("PcrKey")["Procedures"].apply(list).reset_index(name="Procedures")
df_arrest_resus_lists = df_resuscitation.groupby("PcrKey")["Resuscitation"].apply(list).reset_index(name="Resuscitation")
df_arrest_ROSC_lists = df_ROSC.groupby("PcrKey")["ROSC"].apply(list).reset_index(name="ROSC")

## Feature Engineering and Analytical Variable Construction

Raw NEMSIS variables were transformed into analysis-ready features to improve interpretability and support downstream statistical modeling. Feature engineering included creation of binary intervention indicators, aggregation of medication and procedure counts, simplification of cardiac rhythm categories, and construction of clinically meaningful categorical variables.

### Event-Level Feature Construction

Several engineered variables were designed to summarize complex event-level EMS data into patient-level predictors suitable for exploratory analysis and logistic regression modeling.

In [25]:
df_medications_lists.head()

,PcrKey,Medications
0,225614544.0,"[Lactated Ringer's, Epinephrine, Epinephrine]"
1,225614812.0,"[Sodium Bicarbonate, Epinephrine, Sodium Bicar..."
2,225614990.0,"[Epinephrine, Epinephrine, Oxygen, Epinephrine..."
3,225615289.0,"[Epinephrine, Epinephrine, Epinephrine, Epinep..."
4,225615376.0,"[Sodium Bicarbonate, Epinephrine, Sodium Chlor..."


Event-level medication records were grouped by PCR key to capture repeated medications administrations and summarize intervention patterns at the patient-encounter level.

In [26]:
df_medications_lists["meds_unique"] = df_medications_lists["Medications"].apply(get_unique)
df_medications_lists["meds_count"] = df_medications_lists["Medications"].apply(get_count)

df_medications_lists["epi_count"] = df_medications_lists["meds_count"].apply(get_epi_count)
df_medications_lists["epi_count"] = df_medications_lists["epi_count"].fillna(0).astype(int)

df_medications_lists["epi_given"] = df_medications_lists["Medications"].apply(
    lambda x: 1 if isinstance(x, list) and "Epinephrine" in x else 0
).astype(int)

df_medications_lists["total_medications"] = df_medications_lists["Medications"].apply(total_count)
df_medications_lists["unique_medications"] = df_medications_lists["meds_unique"].apply(total_count)

In [27]:
df_procedures_lists["Procedures"] = df_procedures_lists["Procedures"].apply(
    lambda x: ["Unknown Procedure" if pd.isna(i) else i for i in x]
)
df_procedures_lists.head()

,PcrKey,Procedures
0,225614118.0,[Assessment]
1,225614544.0,"[Defibrillation, CPR, Assessment, CPR, IO Inse..."
2,225614812.0,"[Cardiac Monitoring, CPR, Orotracheal Intubati..."
3,225614990.0,"[Assessment, Assessment, Orotracheal Intubatio..."
4,225615289.0,"[Unknown Procedure, Airway Suction, CPR, Cardi..."


Procedure records were similarly aggregated by PCR key to summarize repeated EMS interventions occuring during the same cardiac arrest encounter. Aggregated procedure lists were subsequently used to engineer intervention-specific indicators and overall intervention intensity measures. Unknown or unrecorded intervention entries were retained as explicit categories during aggregation to preserve total intervention counts and overall resuscitation intensity measures.

In [28]:
df_procedures_lists["procedures_unique"] = df_procedures_lists["Procedures"].apply(get_unique)
df_procedures_lists["procedures_count"] = df_procedures_lists["Procedures"].apply(get_count)
df_procedures_lists["defibrillation_status"] = df_procedures_lists["Procedures"].apply(
    lambda x: 1 if isinstance(x, list) and "Defibrillation" in x else 0
).astype(int)
df_procedures_lists["CPR_status"] = df_procedures_lists["Procedures"].apply(
    lambda x: 1 if isinstance(x, list) and "CPR" in x else 0
).astype(int)

df_procedures_lists["total_procedures"] = df_procedures_lists["Procedures"].apply(total_count)
df_procedures_lists["unique_procedures"] = df_procedures_lists["procedures_unique"].apply(total_count)

In [29]:
df_arrest_resus_lists.head()

,PcrKey,Resuscitation
0,225614118.0,[Initiated Chest Compressions]
1,225614544.0,[Initiated Chest Compressions]
2,225614812.0,"[Attempted Ventilation, Initiated Chest Compre..."
3,225614990.0,"[Initiated Chest Compressions, Attempted Venti..."
4,225615289.0,"[Attempted Ventilation, Initiated Chest Compre..."


Resuscitation-related event records were also aggregated by PCR key and used to derive CPR and ventilation-related intervention indicators.

In [30]:
df_arrest_resus_lists["Resuscitation"] = df_arrest_resus_lists["Resuscitation"].map(get_unique)
df_arrest_ROSC_lists["ROSC"] = df_arrest_ROSC_lists["ROSC"].map(get_unique)

### Outcome Variable Encoding

The ROSC outcome variable was converted into a binary classification target for predictive modeling. Cases with documented return of spontaneous circulation were encoded as 1, while cases without ROSC were encoded as 0.

In [31]:
df_arrest_ROSC_lists["ROSC"] = df_arrest_ROSC_lists["ROSC"].apply(ROSC_status)

## Integrated Analytical Dataset Construction

The cleaned and engineered event-level tables were merged into a unified patient-level analytical dataset using the PCR key as the primary identifier. This final dataset combined response, initial acuity, rhythm, intervention, medication, and outcome-related features for downstream exploratory analysis and predictive modeling.

In [32]:
dfs = [df_clean, df_medications_lists, df_procedures_lists, df_arrest_resus_lists,
       df_arrest_ROSC_lists, df_symptoms]

df_final = reduce(lambda left, right: pd.merge(left, right, on="PcrKey", how="left"), dfs)

### Patient-Level Feature Construction

Following table aggregation and dataset integration, additional patient-level composite features were constructed to summarize overall intervention burden and resuscitation intensity during each cardiac arrest encounter.

In [33]:
df_final["intervention_intensity"] = df_final["total_medications"] + df_final["total_procedures"]

### Feature Selection and Column Reduction

Columns that were not clinically relevant, redundant, excessively sparse, or not applicable to downstream analysis were removed to simplify the analytical dataset and improve model interpretability.

In [34]:
df_final.drop(columns=["Medications", "Procedures", "meds_count", "procedures_count",
                       "meds_unique", "procedures_unique", "Resuscitation"], inplace=True)

### Handling Remaining Missing Values

After dataset intergration and feature engineering, remaining missing values were reviewed and handled according to variable type and analytical relevance. Explicit unknown categories were retained where clinically meaningful, while incomplete records affecting model integrity were removed or standardized.

In [35]:
binary_cols = [
    "ROSC",
    "CPR_status",
    "epi_given",
    "intervention_intensity",
    "epi_count",
    "defibrillation_status",
    "total_procedures",
    "unique_procedures",
    "total_medications",
    "unique_medications"
]

for col in binary_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0).astype(int)

## Final Analytical Dataset

The final analytical dataset combined engineered patient-level features, intervention summaries, rhythm classifications, and ROSC outcomes into a unified structure suitable for exploratory analysis and predictive modeling.

In [36]:
df_final.head()

,PcrKey,Response_Type,Arrest,Arrest_Etiology,AED_Prior_to_EMS,Initial_Rhythm,End_of_EMS_Cardiac_Event,Initial_Acuity,Final_Acuity,epi_count,epi_given,total_medications,unique_medications,defibrillation_status,CPR_status,total_procedures,unique_procedures,ROSC,Symptoms,intervention_intensity
0,225614118.0,Ground,"Yes, Prior to Any EMS Arrival",Cardiac Etiology (presumed),No,Asystole,Expired in the Field,NaN,NaN,0,0,0,0,0,0,1,1,0,Cardiovascular,0
1,225614544.0,Ground,"Yes, Prior to Any EMS Arrival",Cardiac Etiology (presumed),No,NaN,Expired in the Field,NaN,NaN,2,1,3,2,1,1,5,4,0,Cardiovascular,8
2,225614812.0,Ground,"Yes, Prior to Any EMS Arrival",Cardiac Etiology (presumed),NaN,NaN,NaN,Emergent (Yellow),Emergent (Yellow),8,1,10,2,0,1,5,5,0,Symptoms/Unknown,15
3,225614990.0,Ground,"Yes, Prior to Any EMS Arrival",Cardiac Etiology (presumed),No,PEA,Expired in the Field,Critical (Red),Critical (Red),4,1,9,5,0,0,4,3,0,Cardiovascular,13
4,225615289.0,Ground,"Yes, Prior to Any EMS Arrival",Cardiac Etiology (presumed),No,Asystole,Expired in ED,Critical (Red),Critical (Red),8,1,8,1,0,1,6,6,0,Cardiovascular,14


In [37]:
df_final.shape

(274531, 20)

In [38]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 274531 entries, 0 to 274530
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   PcrKey                    274531 non-null  float64
 1   Response_Type             274531 non-null  str    
 2   Arrest                    274531 non-null  str    
 3   Arrest_Etiology           274531 non-null  str    
 4   AED_Prior_to_EMS          264370 non-null  str    
 5   Initial_Rhythm            208798 non-null  str    
 6   End_of_EMS_Cardiac_Event  257554 non-null  str    
 7   Initial_Acuity            210301 non-null  str    
 8   Final_Acuity              133267 non-null  str    
 9   epi_count                 274531 non-null  int64  
 10  epi_given                 274531 non-null  int64  
 11  total_medications         274531 non-null  int64  
 12  unique_medications        274531 non-null  int64  
 13  defibrillation_status     274531 non-null  int64  
 14 

## Transition to Analysis

The cleaned and engineered dataset generated in this notebook was subsequently used for exploratory analysis, statistical evaluation, and predictive modeling in [02_analysis.ipynb](./02_analysis.ipynb). The final analytical dataset was exported as a CSV file for downstream analysis and reproducibility.

In [40]:
df_final.to_csv("../outputs/cleaned_data.csv", index=False)